# 25 · Checkpoint: pick a GPU and parallelism for an SLO
Level L4 · Part 3 · core · ~25 min · builds on 21, 22, 23

Time: about 25 min. No GPU and no key are needed. Every number is arithmetic from the model's size, the cards' spec sheets and the link speeds of lessons 19 to 24, so the cells run on a laptop.

## Goal
By the end of this checkpoint you can choose the GPU type, the tensor-parallel degree and the replica count for the 70B model against a stated latency target and a peak request rate. Every number in your decision has its arithmetic printed next to it.

## Why it matters
- **Without a checkpoint:** you pick a card from a blog post, or the cheapest card that loads. The first load test then decides, and a single replica goes down with its only node.
- **With it:** one target, a list of options, and a decision with the arithmetic shown. The result is checkable: change one number and the choice can change.
- **When you don't need it:** a model that fits one card with room for many requests. Then pick the card, add replicas and skip the grid.

## The idea
A capacity plan joins four lessons. Lesson 19 says which splits fit the weights. Lesson 22 prices the all-reduce of each split. Lesson 23 turns one group into replicas for a request rate. Lesson 24 gives each card's memory, bandwidth and FP8 support. This checkpoint runs every **option** (a card, a weight format and a tensor-parallel degree) through those four steps. An option stays only if it meets the latency target. The survivors are ranked, by GPU count when no prices are entered, by cost when they are.

Think of choosing a fleet of delivery vans. The load (the model's weights), the road (the link) and the number of vans (replicas) all matter. The cheapest fleet is the one that meets the schedule at the lowest price. Fewest vans is only a stand-in for cost until prices are entered.

```mermaid
flowchart LR
  O["every option: card x format x TP"] --> F{"weights fit, cache room left?"}
  F -->|no| X["drop"]
  F -->|yes| L{"TTFT p50 at most 0.5 s?"}
  L -->|no| X
  L -->|yes| R["replicas for 6 requests/s x 1.3, at least 2"]
  R --> D["rank: fewest GPUs, or lowest cost"]
```

## Requirements
- [ ] The SLO is written as numbers. The desk's SLO (assumed for this course, not a standard): TTFT p50 at most 0.5 s for a 1,600-token prompt, 6 requests per second at peak, 30% headroom, and at least 2 replicas so one can fail.
- [ ] Each option shows its cache room, batch, prefill time, decode time, requests per second per replica, replica count and GPU count.
- [ ] Options that miss the TTFT target are removed, and the printed numbers say why.
- [ ] The chosen option is the lowest-cost one if you enter prices, otherwise the one with the fewest GPUs.
- [ ] The decision is one paragraph that names its assumptions and the number that would change it.

## Build it
### Step 1: the setup, copied from lessons 19 to 24
The numbers and helpers are the same as in those lessons, so the notebook stands alone. The fixed cost of an all-reduce and the 30% headroom are assumptions, labelled. The card prices are empty: enter yours from your billing catalogue (lesson 24), or leave them `None` and the ranking uses GPU count.

In [1]:
import math

GIB = 2**30
UTIL = 0.92                              # vLLM default gpu_memory_utilization (lesson 19)
P = 70_553_706_496                       # Llama 3.1 70B parameters (lesson 19)
KV = 327_680                             # FP16 cache per token, 320 KiB, all GPUs together (lesson 03)
ACT = 8192 * 2                           # one token's activation vector, 16 KiB (lesson 22)
ALLREDUCES = 160                         # two per layer, 80 layers (lesson 22)
ALLREDUCE_FIXED = 20e-6                  # ASSUMPTION: fixed cost of one all-reduce call (lesson 22)
Q_HEADS, KV_HEADS = 64, 8                # attention heads and KV heads of the 70B model (lesson 19)
PROMPT, ANSWER = 1_600, 200              # a Brightside request (lesson 19)
CTX = PROMPT + ANSWER                    # cache tokens per request
TARGET_RPS, HEADROOM, MIN_REPLICAS = 6, 1.3, 2   # the desk's peak; ASSUMPTION: 30% spare; two replicas so one can fail
TTFT_SLO = 0.5                           # p50 time to first token, seconds: the desk's target
# Spec sheets as in lesson 24 (dense FLOP/s); link = one direction (lesson 22). fp8 None: no FP8 tensor cores.
CARDS = {
    "L4":        dict(mem=24e9,  bw=300e9,  link=32e9,  fp16=121e12, fp8=242e12,  price=None),
    "A100 80GB": dict(mem=80e9,  bw=2039e9, link=300e9, fp16=312e12, fp8=None,    price=None),
    "H100 80GB": dict(mem=80e9,  bw=3350e9, link=450e9, fp16=989e12, fp8=1979e12, price=None),
    "H200":      dict(mem=141e9, bw=4800e9, link=450e9, fp16=989e12, fp8=1979e12, price=None),
}
FORMATS = {"FP16": 2, "FP8": 1}          # bytes per weight
print("cards:", ", ".join(CARDS), "| formats:", ", ".join(FORMATS))

cards: L4, A100 80GB, H100 80GB, H200 | formats: FP16, FP8


### Step 2: the helper that prices one option
`plan` is the lesson 23 arithmetic in one function. It refuses a degree that does not divide the heads (lesson 19) and an option whose weights do not fit. Run the refusal first, so you see what it prints.

In [2]:
def plan(card, fmt, tp):
    """One option: cache room, batch, the two costs per request, replicas and GPUs."""
    c, bpp = CARDS[card], FORMATS[fmt]
    if Q_HEADS % tp or KV_HEADS % tp:
        raise ValueError(f"TP={tp} must divide {Q_HEADS} query and {KV_HEADS} KV heads")
    room = c["mem"] * UTIL - P * bpp / tp
    if room <= 0:
        raise ValueError(f"{card} {fmt} TP={tp}: weights do not fit ({room / GIB:.1f} GiB room)")
    tokens = room * tp / KV
    batch = int(tokens // CTX)
    if batch < 1:
        raise ValueError(f"{card} {fmt} TP={tp}: no room for one request")
    maths = c["fp8"] if fmt == "FP8" and c["fp8"] else c["fp16"]   # no FP8 tensor cores: FP16 maths
    def ar(n):                                                    # one all-reduce of n tokens
        return 0.0 if tp == 1 else 2 * (tp - 1) / tp * n * ACT / c["link"] + ALLREDUCE_FIXED
    prefill = 2 * P * PROMPT / (tp * maths) + ALLREDUCES * ar(PROMPT)
    step = (P * bpp / tp + batch * CTX * KV / tp) / c["bw"] + ALLREDUCES * ar(batch)
    # the batch shares every step, so one request pays 1/batch of its ANSWER steps (this sets requests per second)
    decode = ANSWER * step / batch
    rps = 1 / (prefill + decode)
    replicas = max(MIN_REPLICAS, math.ceil(TARGET_RPS * HEADROOM / rps))
    return dict(card=card, fmt=fmt, tp=tp, room_gib=room / GIB, tokens=tokens, batch=batch, prefill=prefill,
                decode=decode, rps=rps, replicas=replicas, gpus=replicas * tp, ttft_ok=prefill <= TTFT_SLO)
print("plan(card, format, tp) ready for", len(CARDS), "cards")

plan(card, format, tp) ready for 4 cards


Now the refusal. Run one option that cannot fit: FP16 weights split four ways on L4 cards.

In [3]:
plan("L4", "FP16", 4)   # FP16 weights split 4 ways: 32.8 GiB per L4, but one L4 gives 20.56 GiB

ValueError: L4 FP16 TP=4: weights do not fit (-12.3 GiB room)

**What you should see:** a `ValueError`, because the weights do not fit. Lesson 19 found the same answer. The fix is a smaller format or more cards per group.

### Step 3: every option, with its arithmetic
Each card, each format and each degree 1, 2, 4 or 8 is one option. The loop keeps the options that fit and prints the ones that meet the latency target. The first print is the best option per card: fewest GPUs, then the lower prefill.

In [4]:
options = []
for card in CARDS:
    for fmt in FORMATS:
        for tp in (1, 2, 4, 8):
            try:
                options.append(plan(card, fmt, tp))
            except ValueError:
                pass                                              # does not fit: not an option
ok = [o for o in options if o["ttft_ok"]]
print(f"{len(options)} options fit in memory; {len(ok)} meet TTFT p50 at most {TTFT_SLO} s")
for o in options:
    if not o["ttft_ok"]:
        print(f"dropped by TTFT: {o['card']} {o['fmt']} TP={o['tp']}, prefill {o['prefill']:.3f} s (target {TTFT_SLO} s)")
for card in CARDS:
    best = min((o for o in ok if o["card"] == card), key=lambda o: (o["gpus"], o["prefill"]), default=None)
    if best is None:
        print(f"{card:10} no option meets the target")
        continue
    print(f"{card:10} {best['fmt']} TP={best['tp']}: batch {best['batch']}, prefill {best['prefill']:.3f} s, "
          f"decode {best['decode']:.3f} s, {best['rps']:.2f} req/s per replica, "
          f"{best['replicas']} replicas x {best['tp']} = {best['gpus']} GPUs")

24 options fit in memory; 23 meet TTFT p50 at most 0.5 s
dropped by TTFT: A100 80GB FP8 TP=1, prefill 0.724 s (target 0.5 s)
L4         FP8 TP=8: batch 179, prefill 0.349 s, decode 0.114 s, 2.16 req/s per replica, 4 replicas x 8 = 32 GPUs
A100 80GB  FP16 TP=4: batch 259, prefill 0.205 s, decode 0.033 s, 4.20 req/s per replica, 2 replicas x 4 = 8 GPUs
H100 80GB  FP8 TP=2: batch 129, prefill 0.070 s, decode 0.040 s, 9.12 req/s per replica, 2 replicas x 2 = 4 GPUs
H200       FP8 TP=1: batch 100, prefill 0.114 s, decode 0.054 s, 5.95 req/s per replica, 2 replicas x 1 = 2 GPUs


**What you should see:** the number of options that fit and the number that meet the 0.5 s target, then the option dropped by the target, then one line per card. Read the L4 line against the H100 line: the L4 needs more cards per request, because its memory and bandwidth are smaller (lesson 24). `A100 80GB` FP8 has no FP8 tensor cores, so its FP8 option is priced with FP16 maths: it would show the FP16 speed with half the weight bytes, and its prefill misses the target, so the filter drops it and it never wins.

### Step 4: your decision
**Your task:** write `chosen`, one option from `ok` (a dict from `plan`). Rank by GPU count, and break a tie with the lower prefill time. If you entered prices in `CARDS`, rank by price times GPUs; write the ranking so it falls back to GPU count while prices are None. Print one sentence for the design review.

In [5]:
# your code here

## Reference solution
**Try step 4 first.** The answer is in the hidden cell below.

In [6]:
priced = all(CARDS[o["card"]]["price"] is not None for o in ok)
key = (lambda o: (CARDS[o["card"]]["price"] * o["gpus"], o["prefill"])) if priced else (lambda o: (o["gpus"], o["prefill"]))
chosen = min(ok, key=key)
assert chosen["ttft_ok"] and chosen["replicas"] >= MIN_REPLICAS
print("ranked by:", "price times GPUs" if priced else "GPU count (a proxy: enter prices in CARDS to rank by cost)")
print(f"chosen: {chosen['card']}, {chosen['fmt']} weights, TP={chosen['tp']}, "
      f"{chosen['replicas']} replicas, {chosen['gpus']} GPUs in total")
print(f"cache room per replica: {chosen['room_gib']:.1f} GiB")

ranked by: GPU count (a proxy: enter prices in CARDS to rank by cost)
chosen: H200, FP8 weights, TP=1, 2 replicas, 2 GPUs in total
cache room per replica: 55.1 GiB


**The decision, with this run's numbers.** Two H200s with FP8 weights, one card per replica (TP=1). Each replica has 55 GiB of cache room, a batch of 100 requests and a prefill of 0.114 s, which is under the 0.5 s target. One replica does 5.95 requests per second, so 6 × 1.3 = 7.8 needs two replicas: 2 GPUs in total.

**What the arithmetic leaves out.** It assumes no queueing (lesson 26 covers utilisation and queue waits) and full batches. It also uses the 20 µs fixed cost per all-reduce, an assumption carried from lesson 22. That cost only touches TP>1 options, so the TP=1 H200 choice does not depend on it. The 0.5 s target and the 2-replica floor are the desk's numbers; a different desk would change them. Measure before you buy (lesson 27), and read the cost per million tokens (lesson 28) before you commit.

**Where the price decides.** The GPU-count proxy picks H200 because it needs the fewest cards. That is not a cost result. Four H100s in two replicas of TP=2 also meet the target. H200 wins on cost only if one H200 costs less than twice one H100 (4 × H100 < 2 × H200 needs H100 < H200 / 2 for the H100 fleet to win). Enter the prices in `CARDS` and rerun the reference cell to see which one wins.

## Key terms
- **SLO (service level objective)** — the target a service is measured against, here TTFT p50 at most 0.5 s.
- **Headroom** — spare capacity kept above the peak load, here 30%, for bursts and for maintenance.
- **Replica** — one full copy of a serving group behind a load balancer (lesson 23).
- **Option** — one choice of card, weight format and tensor-parallel degree that this checkpoint prices.

**Interview:** "Pick a GPU and parallelism for a 70B model at 6 requests per second with a 500 ms time to first token." Strong answer: keep the options whose weights fit with cache room. Drop those whose prefill, with its all-reduces, misses 500 ms. Count replicas from requests per second per replica with 30% headroom and at least two replicas. Rank by cost per hour, and name the assumptions that would change the answer.